In [3]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize


class BlendError(Exception):
  
    pass


STREAM_DATA = pd.DataFrame([
    {"component": "Isomerate",            "max_flow": 95.0,  "ron": 86.0,  "b_value": 1.0, "sulfur_ppm": 0.0},
    {"component": "Reformate",             "max_flow": 80.0,  "ron": 98.0,  "b_value": 1.0, "sulfur_ppm": 0.0},
    {"component": "Prime G R/D (Cracked)", "max_flow": 105.0, "ron": 86.8,  "b_value": 1.4, "sulfur_ppm": 10.0},
    {"component": "CCRU NHT DSN",          "max_flow": 25.0,  "ron": 60.0,  "b_value": 1.0, "sulfur_ppm": 0.5},
    {"component": "OHCU Lt. Naphtha",      "max_flow": 17.5,  "ron": 72.0,  "b_value": 1.0, "sulfur_ppm": 5.0},
    {"component": "Iso Octene (Polymers)", "max_flow": 8.5,   "ron": 115.0, "b_value": 2.6, "sulfur_ppm": 45.0},
    {"component": "OHCU Hy. Naphtha",      "max_flow": 2.5,   "ron": 60.0,  "b_value": 1.0, "sulfur_ppm": 8.0},
])


def calculate_blend(flows):
    
    df = STREAM_DATA.copy()
    df["flow"] = [flows.get(name, 0.0) for name in df["component"]]

    
    for _, row in df.iterrows():
        if row["flow"] < 0:
            raise BlendError(f"{row['component']}: flow cannot be negative.")
        if row["flow"] > row["max_flow"]:
            raise BlendError(
                f"{row['component']}: flow {row['flow']} exceeds max design "
                f"flow of {row['max_flow']}."
            )

    total_flow = df["flow"].sum()
    if total_flow == 0:
        raise BlendError("Total flow is zero — nothing to blend.")

    df["vol_fraction"] = df["flow"] / total_flow

    # --- RON
    top = (df["ron"] * df["b_value"] * df["vol_fraction"]).sum()
    bottom = (df["b_value"] * df["vol_fraction"]).sum()
    blend_ron = top / bottom

    # --- Sulfur
    blend_sulfur = (df["sulfur_ppm"] * df["vol_fraction"]).sum()

    return {
        "blend_ron": blend_ron,
        "blend_sulfur_ppm": blend_sulfur,
        "total_flow": total_flow,
        "details": df[df["flow"] > 0][["component", "flow", "vol_fraction"]],
    }


def optimize_for_ron(target_ron, total_throughput, max_sulfur_ppm=10.0):
   
    df = STREAM_DATA
    ron = df["ron"].values
    b_value = df["b_value"].values
    sulfur = df["sulfur_ppm"].values
    max_flow = df["max_flow"].values
    n = len(df)

    if total_throughput > max_flow.sum():
        raise BlendError(
            f"Requested throughput ({total_throughput}) is more than all "
            f"streams combined can supply ({max_flow.sum():.1f})."
        )

    def blend_ron(x):
        return np.sum(ron * b_value * x) / np.sum(b_value * x)

    def blend_sulfur(x):
        return np.sum(sulfur * x) / np.sum(x)


    constraints = [
        {"type": "eq", "fun": lambda x: np.sum(x) - total_throughput},
        {"type": "eq", "fun": lambda x: blend_ron(x) - target_ron},
        {"type": "ineq", "fun": lambda x: max_sulfur_ppm - blend_sulfur(x)},
    ]

    bounds = [(0.0, mf) for mf in max_flow]

    even_split = total_throughput / n

    def objective(x):
        return np.sum((x - even_split) ** 2)

    starting_guess = np.full(n, even_split)
    starting_guess = np.minimum(starting_guess, max_flow)

    result = minimize(
        objective, starting_guess, method="SLSQP",
        bounds=bounds, constraints=constraints,
        options={"maxiter": 500, "ftol": 1e-9},
    )

    achieved_ron = blend_ron(result.x)
    achieved_sulfur = blend_sulfur(result.x)

    ok = (
        result.success
        and abs(achieved_ron - target_ron) <= 0.01
        and achieved_sulfur <= max_sulfur_ppm + 1e-6
    )

    if not ok:
        raise BlendError(
            f"Could not hit RON {target_ron} at {total_throughput} m3/hr "
            f"within the sulfur/flow limits. Best attempt: RON="
            f"{achieved_ron:.2f}, Sulfur={achieved_sulfur:.2f} ppm."
        )

    details = df.copy()
    details["optimized_flow"] = result.x
    details["vol_fraction"] = result.x / result.x.sum()

    return {
        "achieved_ron": achieved_ron,
        "achieved_sulfur_ppm": achieved_sulfur,
        "total_flow": result.x.sum(),
        "details": details[["component", "optimized_flow", "vol_fraction"]],
    }


if __name__ == "__main__":
    pd.set_option("display.float_format", lambda v: f"{v:0.3f}")

    # --- Example 1: forward calculation ---
    print("\n--- Forward Blend Calculation ---")
    my_flows = {
        "Isomerate": 60.0, "Reformate": 55.0, "Prime G R/D (Cracked)": 70.0,
        "CCRU NHT DSN": 15.0, "OHCU Lt. Naphtha": 10.0,
        "Iso Octene (Polymers)": 5.0, "OHCU Hy. Naphtha": 2.0,
    }
    result = calculate_blend(my_flows)
    print(f"Total Flow   : {result['total_flow']:.3f} m3/hr")
    print(f"Blend RON    : {result['blend_ron']:.3f}")
    print(f"Blend Sulfur : {result['blend_sulfur_ppm']:.3f} ppm")
    print(result["details"].to_string(index=False))

    # --- Example 2: error handling ---
    print("\n--- Error Handling ---")
    try:
        calculate_blend({"Reformate": 999.0})
    except BlendError as e:
        print(f"Correctly caught error: {e}")

    # --- Example 3: optimize for target RON = 91 ---
    print("\n--- Optimize for Target RON = 91 ---")
    plan = optimize_for_ron(target_ron=91.0, total_throughput=250.0)
    print(f"Achieved RON    : {plan['achieved_ron']:.3f}")
    print(f"Achieved Sulfur : {plan['achieved_sulfur_ppm']:.3f} ppm")
    print(plan["details"].to_string(index=False))

    # --- Example 4: an impossible target ---
    print("\n--- Impossible Target ---")
    try:
        optimize_for_ron(target_ron=99.5, total_throughput=250.0)
    except BlendError as e:
        print(f"Correctly caught error: {e}")


--- Forward Blend Calculation ---
Total Flow   : 217.000 m3/hr
Blend RON    : 88.108
Blend Sulfur : 4.601 ppm
            component   flow  vol_fraction
            Isomerate 60.000         0.276
            Reformate 55.000         0.253
Prime G R/D (Cracked) 70.000         0.323
         CCRU NHT DSN 15.000         0.069
     OHCU Lt. Naphtha 10.000         0.046
Iso Octene (Polymers)  5.000         0.023
     OHCU Hy. Naphtha  2.000         0.009

--- Error Handling ---
Correctly caught error: Reformate: flow 999.0 exceeds max design flow of 80.0.

--- Optimize for Target RON = 91 ---
Achieved RON    : 91.000
Achieved Sulfur : 4.686 ppm
            component  optimized_flow  vol_fraction
            Isomerate          74.733         0.299
            Reformate          80.000         0.320
Prime G R/D (Cracked)          71.025         0.284
         CCRU NHT DSN           0.000         0.000
     OHCU Lt. Naphtha          15.743         0.063
Iso Octene (Polymers)           8.500  